# PredictGuard — Phase 1, Stage 3: Feature Engineering

**Project**: Explainable Predictive Maintenance System  
**Dataset**: Microsoft Azure Predictive Maintenance  
**Author**: PredictGuard Contributors  
**Stage**: 3 of 6 — Feature Engineering

---

## Objective

Construct **leakage-safe, domain-driven features** for predictive maintenance.  
All reusable logic lives in `src/feature_engineering.py` and automated leakage checks live in `src/validators/leakage_checker.py`.

### Engineered Feature Sets
1. **Trailing Rolling Statistics**: 3h & 24h trailing mean, std, min, max, median, range
2. **Rate of Change**: 1h, 3h, 6h, 12h deltas, velocity, acceleration, percentage change
3. **Machine Normal Deviation**: Expanding z-score shifted by 1 per machine
4. **Interaction Terms**: Domain-inspired sensor products & ratios (safe division)
5. **Error History**: Trailing 24h & 72h error counts, days since last error, error type counts
6. **Maintenance History**: Trailing 30d, 90d, 1y maintenance counts, days since last maintenance
7. **Static Specifications**: Machine age & one-hot model specs

---
## 0. Environment Setup & Imports

In [ ]:
import logging
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# Make src/ importable
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import target_creation as tc
from src import feature_engineering as fe
from src.validators import leakage_checker as lc

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("stage3_notebook")
logger.info("Stage 3 notebook started.")

In [ ]:
RAW_DIR = project_root / "data" / "raw"
PROCESSED_DIR = project_root / "data" / "processed"
REPORTS_DIR = project_root / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

---
## 1. Load Datasets

Load `telemetry_with_targets.parquet` from Stage 2 alongside raw error, maintenance, and machine metadata tables.

In [ ]:
df_targets = pd.read_parquet(PROCESSED_DIR / "telemetry_with_targets.parquet")
datasets = tc.load_stage1_data(RAW_DIR)
errors_df = datasets["errors"]
maint_df = datasets["maint"]
machines_df = datasets["machines"]

print(f"Loaded target dataset shape: {df_targets.shape}")

---
## 2. Compute Trailing Rolling Statistics

> **Why trailing windows?**  
> Centered rolling windows look into future timestamps $t+1 \dots t+k$, introducing data leakage. Trailing windows compute statistics strictly over $[t-W+1, t]$ using available past data only.

In [ ]:
df_features = fe.compute_rolling_features(df_targets, windows=[3, 24])
print(f"Columns after rolling stats: {df_features.shape[1]}")

---
## 3. Rate of Change & Shifted Expanding Z-Scores

> **Why shift expanding statistics?**  
> Expanding mean and std must be shifted by 1 timestep so that observation $x(t)$ is excluded from its own baseline $z = (x(t) - \mu_{t-1}) / \sigma_{t-1}$.

In [ ]:
df_features = fe.compute_rate_of_change_features(df_features, lags=[1, 3, 6, 12])
df_features = fe.compute_expanding_zscores(df_features)
df_features = fe.compute_interaction_features(df_features)
print(f"Columns after rate & z-score features: {df_features.shape[1]}")

---
## 4. Error, Maintenance & Machine Features

In [ ]:
df_features = fe.compute_error_features(df_features, errors_df)
df_features = fe.compute_maintenance_features(df_features, maint_df)
df_features = fe.compute_static_machine_features(df_features, machines_df)
print(f"Final feature dataset shape: {df_features.shape}")

---
## 5. Automated Leakage Audit & Feature Diagnostics

In [ ]:
leakage_audit = lc.run_full_leakage_suite(df_features)
val_report = fe.validate_engineered_features(df_features)

print(f"\nLeakage Audit Status: {'✅ PASSED' if leakage_audit['overall_leakage_free'] else '❌ FAILED'}")
print(f"Feature Validation Status: {'✅ PASSED' if val_report['validation_passed'] else '⚠️ FLAGGED'}")

---
## 6. Feature Importance Preview (Mutual Information & Correlation)

In [ ]:
preview_df = fe.compute_feature_importance_preview(df_features, target_col="y_failure", n_sample=50_000)
display(preview_df.head(15))

---
## 7. Persist Feature Outputs & Feature Dictionary

In [ ]:
df_features.to_parquet(PROCESSED_DIR / "features.parquet", index=False, engine="pyarrow")
df_features.to_csv(PROCESSED_DIR / "features.csv", index=False)

feature_dict_df = fe.generate_feature_dictionary(df_features)
feature_dict_df.to_csv(PROCESSED_DIR / "feature_dictionary.csv", index=False)

print("✅ Saved features.parquet, features.csv, and feature_dictionary.csv under data/processed/")

---
## 8. Generate Visualisations

In [ ]:
fe.plot_rolling_stats_example(df_features, FIGURES_DIR)
fe.plot_expanding_zscore_example(df_features, FIGURES_DIR)
fe.plot_rate_of_change(df_features, FIGURES_DIR)
fe.plot_maintenance_timeline(maint_df, FIGURES_DIR)
fe.plot_error_timeline(errors_df, FIGURES_DIR)
fe.plot_feature_correlation_heatmap(df_features, FIGURES_DIR)
fe.plot_top_feature_importance(preview_df, FIGURES_DIR)

print("✅ All Stage 3 figures generated and saved under reports/figures/.")

---
## 9. Stage 3 Summary & Next Steps

| Deliverable | Status |
|---|---|
| Trailing Rolling Features | ✅ Completed |
| Rate of Change Features | ✅ Completed |
| Shifted Expanding Z-Scores | ✅ Completed |
| Error & Maintenance History | ✅ Completed |
| Automated Leakage Audit | ✅ 100% Leakage-Free |
| Feature Parquet & CSV | ✅ Saved to `data/processed/` |
| Feature Dictionary | ✅ Saved to `data/processed/feature_dictionary.csv` |
| Markdown Report | ✅ Saved to `reports/feature_engineering_report.md` |

### Ready for Stage 4 (Train/Test Splitting & Modelling)